# Step 0.5 の 6: Gemma 4 E2B-it が T4 の llama-server で起動するか（docs/v2-light-model-plan.md §3・§4）

Gemma 4 E2B-it は T4 の vLLM では起動しなかった（#7）。表の条件付き候補なので、**llama-server（GGUF）で起動するか**だけを先に確かめる。起動したときだけ Step 2 と同じ線で測る（測る仕組みは別に作る）。

- 重み: `ggml-org/gemma-4-E2B-it-GGUF` の **Q8_0**（4.63 GiB・revision 固定）。BF16（8.67 GiB）は 8GB 予算に重みだけで入らず、T4 は bf16 の計算部品が無いので使わない。Q8_0 は 8bit で、手を加えない重みに一番近い
- 実行系: llama.cpp の配布版（ubuntu・CUDA 12.8・タグ固定）。T4 で動かなければ、同じタグをソースから T4 向け（sm_75）に作り直してもう一度だけ試す
- 文脈長 4096（Step 2 の 2 候補は必要な長さが 3,584 だった。Gemma の数え方は違うので、長い問で入力のトークン数を記録する）
- 判定は 2 つに分ける。**GPU で動く**＝起動・CUDA の装置が見える・全層が GPU に載る・T4 用の部品が無いという誤りが出ない・重みの一部が GPU に置かれたと記録に出る。**答えが使える**＝3 問とも日本語で答え、考える文が混ざらない。ソースからの作り直しは前者が落ちたときだけ
- **2 回目（10/23）**: 1 回目は考える文が既定で出て、記録も控えめで載り方が読めなかった。考える機能を切る指定（`--reasoning off`）と詳しい記録（`-v`）を足し、GPU の判定から「4,000 MiB 以上増える」を外した（入力側の埋め込みは llama.cpp が必ず CPU に置くので、前提が誤り）。答えの判定は変えていない。数えない問を 1 つ先に流す
- 確かめる事: 起動する・全層が GPU に載る・日本語で答える・考える文が混ざらない・資料 2,800 字（長い入力）に答える・GPU の最大使用量（8GB = 7,680 MiB、12GB = 11,776 MiB と比べる）
- llama.cpp は vLLM の GPU_UTIL のような上限を掛けられないので、実測の最大値で判定する（計画 §4）
- データは架空の社内規程（evaluation/corpus）だけ。トークン不要。ランタイムは **T4 GPU**。「すべてのセルを実行」で流れる

In [ ]:
# 設定
LLAMA_TAG = "b11836"
LLAMA_ASSET = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
CUDART_ASSET = f"cudart-llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz"
GGUF_REPO = "ggml-org/gemma-4-E2B-it-GGUF"
GGUF_REV = "6938c58e698502037401a5f19f6905f2fb2f7dee"
GGUF_FILE = "gemma-4-E2B-it-Q8_0.gguf"
CTX = 4096
PORT = 8001
BUDGETS_MIB = {"8GB": 7680, "12GB": 11776}
REPO = "/content/repo"
OUT = "/content/step05g"

In [ ]:
import json, os, pathlib, re, subprocess, threading, time, urllib.request

OUTP = pathlib.Path(OUT)
if OUTP.exists() and any(OUTP.iterdir()):
    raise RuntimeError(f"{OUT} が空でない（前の結果と混ざるので止める）")
OUTP.mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "redesign/v2",
                "https://github.com/Maaa2005/local-rag-agent.git", REPO], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def sh(cmd, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, **kw)

g = sh(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total,driver_version", "--format=csv,noheader,nounits"]).stdout.strip()
GPU_NAME, CAP, TOTAL_MIB, DRIVER = [x.strip() for x in g.split(",")]
def read_mib():
    out = sh(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"]).stdout
    return int(re.search(r"\d+", out).group())
BASELINE_MIB = read_mib()
CUDA_TOOLKIT = (re.search(r"release ([\d.]+)", sh(["/usr/local/cuda/bin/nvcc", "--version"]).stdout or "") or [None, None])[1]

# GPU の使用量を 0.5 秒ごとに記録する（最大値を区間ごとに切り出す）
GPU_MEM = []
def _poll():
    while True:
        try:
            GPU_MEM.append((time.time(), read_mib()))
        except Exception:
            pass
        time.sleep(0.5)
threading.Thread(target=_poll, daemon=True).start()
def peak(t0, t1):
    xs = [m for t, m in GPU_MEM if t0 <= t <= t1]
    return max(xs) if xs else None
print(f"{GPU_NAME} compute {CAP} 全体 {TOTAL_MIB} MiB / 起動前 {BASELINE_MIB} MiB / driver {DRIVER} / nvcc {CUDA_TOOLKIT} / commit {COMMIT[:7]}")

In [ ]:
# 1. 重みを取る（revision 固定・トークン不要）
from huggingface_hub import hf_hub_download
t = time.time()
GGUF_PATH = hf_hub_download(GGUF_REPO, GGUF_FILE, revision=GGUF_REV, local_dir="/content/gguf")
print(GGUF_PATH, f"{os.path.getsize(GGUF_PATH) / 2**30:.2f} GiB", f"{time.time() - t:.0f} 秒")

In [ ]:
# 2. 試す手順（起動 → 3 問 → 落とす）。配布版でもソースから作った版でも同じ手順を使う
def corpus_text(limit_chars):
    parts = []
    for p in sorted(pathlib.Path(REPO, "evaluation/corpus/general").glob("*.md")):
        parts.append(f"[{p.name}]\n" + p.read_text())
    return "\n\n".join(parts)[:limit_chars]

LONG_CTX = corpus_text(2800)
QUESTIONS = [
    ("short", [{"role": "user", "content": "日本の首都はどこですか。一文で答えてください。"}]),
    ("long", [{"role": "system", "content": "あなたは社内規程に答える係です。与えられた資料だけを根拠に、日本語で簡潔に答えてください。資料に無い事は「資料にありません」と答えてください。"},
              {"role": "user", "content": f"資料:\n{LONG_CTX}\n\n質問: 国内出張の日当はいくらですか。"}]),
    ("long_again", None),   # 同じ長い問をもう一度（2 回目の速さ）
]

def post(msgs, timeout=180):
    body = json.dumps({"model": "gemma", "messages": msgs, "temperature": 0, "max_tokens": 256}).encode()
    req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/chat/completions", body, {"Content-Type": "application/json"})
    t = time.time()
    with urllib.request.urlopen(req, timeout=timeout) as r:
        d = json.loads(r.read())
    return d, time.time() - t

def trial(server_bin, label, env):
    rec = {"label": label, "server_bin": server_bin}
    v = sh([server_bin, "--version"], env=env)
    rec["version"] = (v.stdout + v.stderr).strip()[-300:]
    dev = sh([server_bin, "--list-devices"], env=env)
    rec["devices"] = (dev.stdout + dev.stderr).strip()[-500:]
    print(f"[{label}] version: {rec['version']}\n[{label}] devices: {rec['devices']}")
    log = OUTP / f"server_{label}.log"
    t0 = time.time()
    proc = subprocess.Popen([server_bin, "-m", GGUF_PATH, "-ngl", "999", "-c", str(CTX), "--jinja",
                             "--host", "127.0.0.1", "--port", str(PORT), "-np", "1", "--reasoning", "off", "-v"],
                            stdout=open(log, "w"), stderr=subprocess.STDOUT, start_new_session=True, env=env)
    ready = False
    while time.time() - t0 < 600 and proc.poll() is None:
        try:
            urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3)
            ready = True
            break
        except Exception:
            time.sleep(2)
    rec["started"] = ready
    rec["ready_seconds"] = round(time.time() - t0, 1)
    rec["answers"] = []
    if ready:
        # 1 回目は部品の読み込みで遅いので、数えない問を 1 つ先に流す（秒だけ記録）
        try:
            _, sec = post([{"role": "user", "content": "こんにちは"}])
            rec["warmup_seconds"] = round(sec, 1)
        except Exception as e:
            rec["warmup_error"] = repr(e)[:300]
        last = None
        for name, msgs in QUESTIONS:
            msgs = msgs or last
            last = msgs
            try:
                d, sec = post(msgs)
                m = d["choices"][0]["message"]
                text = m.get("content") or ""
                rec["answers"].append({"name": name, "seconds": round(sec, 2), "prompt_tokens": d["usage"]["prompt_tokens"],
                                       "completion_tokens": d["usage"]["completion_tokens"], "finish": d["choices"][0]["finish_reason"],
                                       "timings": {k: round(v, 1) for k, v in (d.get("timings") or {}).items() if k in ("prompt_per_second", "predicted_per_second")},
                                       "has_reasoning": bool(m.get("reasoning_content")),
                                       "thinking_marks": bool(re.search(r"<\|?channel|<\|?think|thought", text)),
                                       "japanese": bool(re.search(r"[\u3040-\u30ff]", text)), "head": text[:160]})
            except Exception as e:
                rec["answers"].append({"name": name, "error": repr(e)[:300]})
    t1 = time.time()
    rec["peak_mib"] = peak(t0, t1)
    try:
        os.killpg(proc.pid, 9)
    except Exception:
        pass
    proc.wait()
    lines = log.read_text(errors="replace").splitlines()
    rec["log_hits"] = [l[:240] for l in lines if re.search(r"offloaded|buffer size|compute capability|error|ERROR|failed|no kernel image|reasoning|thinking", l)][:40]
    log_all = "\n".join(lines)
    off = re.search(r"offloaded (\d+)/(\d+) layers to GPU", log_all)
    rec["cuda_device"] = bool(re.search(r"CUDA0", rec["devices"]))
    rec["offloaded"] = off.group(0) if off else None
    rec["all_layers_on_gpu"] = bool(off and off.group(1) == off.group(2))
    rec["no_kernel_image"] = "no kernel image" in log_all
    rec["gpu_delta_mib"] = (rec["peak_mib"] - BASELINE_MIB) if rec["peak_mib"] is not None else None
    # 重みのうち GPU と CPU に置かれた量（記録のみ。入力側の埋め込みは llama.cpp が必ず CPU に置く）
    rec["model_buffer_mib"] = {m.group(1): float(m.group(2)) for m in re.finditer(r"(CUDA0|CPU)\S* model buffer size =\s*([\d.]+) MiB", log_all)}
    # 起動の判定（GPU で動いたか）と、答えの判定を分ける。作り直しは前者が落ちたときだけ
    rec["runs_on_gpu"] = bool(ready and rec["cuda_device"] and rec["all_layers_on_gpu"] and not rec["no_kernel_image"]
                              and rec["model_buffer_mib"].get("CUDA0", 0) > 0)
    rec["answers_ok"] = bool(ready and len(rec["answers"]) == 3 and all("error" not in a and a["japanese"] and not a["thinking_marks"] and not a["has_reasoning"] for a in rec["answers"]))
    rec["ok"] = rec["runs_on_gpu"] and rec["answers_ok"]
    for a in rec["answers"]:
        print(f"[{label}]", json.dumps(a, ensure_ascii=False))
    print(f"[{label}] started={ready} ready={rec['ready_seconds']}s peak={rec['peak_mib']} MiB gpu={rec['runs_on_gpu']} ({rec['offloaded']}, cuda={rec['cuda_device']}, +{rec['gpu_delta_mib']} MiB) answers={rec['answers_ok']}")
    print("\n".join(rec["log_hits"]))
    print("--- 記録の最後 60 行")
    print("\n".join(l[:240] for l in lines[-60:]))
    time.sleep(3)
    return rec

In [ ]:
# 3. 配布版（ubuntu・CUDA 12.8）で試す
TRIALS = []
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
pre = pathlib.Path("/content/llama-pre")
pre.mkdir(exist_ok=True)
for a in (LLAMA_ASSET, CUDART_ASSET):
    r = sh(["bash", "-c", f"curl -sfL {base}/{a} | tar xz -C {pre}"])
    print(a, "rc", r.returncode, r.stderr[-300:])
found = sorted(pre.rglob("llama-server"))
if found:
    libdirs = sorted({str(p.parent) for p in pre.rglob("*.so*")})
    env = dict(os.environ, LD_LIBRARY_PATH=":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]))
    TRIALS.append(trial(str(found[0]), "prebuilt", env))
else:
    print("配布版に llama-server が見つからない")
    TRIALS.append({"label": "prebuilt", "started": False, "runs_on_gpu": False, "answers_ok": False, "ok": False, "note": "llama-server が無い"})

In [ ]:
# 4. 配布版が GPU で動かなかったときだけ、同じタグをソースから T4（sm_75）向けに作って 1 回だけ試す
if not TRIALS[-1].get("runs_on_gpu"):
    t = time.time()
    src = "/content/llama.cpp"
    steps = [["git", "clone", "-q", "--depth", "1", "-b", LLAMA_TAG, "https://github.com/ggml-org/llama.cpp.git", src],
             ["cmake", "-S", src, "-B", f"{src}/build", "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=75",
              "-DLLAMA_CURL=OFF", "-DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc"],
             ["cmake", "--build", f"{src}/build", "--target", "llama-server", "-j", str(os.cpu_count() or 2)]]
    for s in steps:
        r = sh(s)
        print(" ".join(s[:3]), "rc", r.returncode, (r.stdout + r.stderr)[-800:])
        if r.returncode != 0:
            break
    print(f"作るのに {time.time() - t:.0f} 秒")
    b = pathlib.Path(f"{src}/build/bin/llama-server")
    if b.exists():
        TRIALS.append(trial(str(b), "source_sm75", dict(os.environ)))
    else:
        TRIALS.append({"label": "source_sm75", "started": False, "runs_on_gpu": False, "answers_ok": False, "ok": False, "note": "作れなかった"})
else:
    print("配布版が GPU で動いたので作り直さない")

In [ ]:
# 5. まとめ
best = next((t for t in TRIALS if t.get("runs_on_gpu")), TRIALS[-1])
pk = best.get("peak_mib")
SUMMARY = {"step": "0.5-6", "gpu": GPU_NAME, "compute_cap": CAP, "total_mib": TOTAL_MIB, "baseline_mib": BASELINE_MIB,
           "driver": DRIVER, "nvcc": CUDA_TOOLKIT, "commit": COMMIT, "llama_tag": LLAMA_TAG,
           "gguf": {"repo": GGUF_REPO, "revision": GGUF_REV, "file": GGUF_FILE}, "ctx": CTX,
           "started": any(t.get("started") for t in TRIALS), "runs_on_gpu": best.get("runs_on_gpu", False),
           "answers_ok": best.get("answers_ok", False), "ok": best.get("ok", False), "used": best.get("label"),
           "peak_note": "文脈 4096・検索なしの仮の値。Step 2 の V の判定ではない",
           "peak_mib": pk, "within": {k: (pk <= v if pk is not None else None) for k, v in BUDGETS_MIB.items()},
           "trials": [{k: v for k, v in t.items() if k != "answers"} | {"answers": [{k2: v2 for k2, v2 in a.items() if k2 != "head"} for a in t.get("answers", [])]} for t in TRIALS]}
(OUTP / "summary.json").write_text(json.dumps(SUMMARY, ensure_ascii=False, indent=2) + "\n")
print("SUMMARY_JSON " + json.dumps(SUMMARY, ensure_ascii=False))